In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# Install haversine jika belum ada
try:
    from haversine import haversine, Unit
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'haversine', '-q'])
    from haversine import haversine, Unit

from sklearn.cluster import DBSCAN, KMeans

# Install folium jika belum ada
try:
    import folium
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'folium', '-q'])
    import folium

pd.options.display.max_columns = None

# 1.5 SETUP ENVIRONMENT (CLONE REPO)
import subprocess

repo_url  = "https://github.com/shineistu86/CPS-CC26.git"
repo_name = "CPS-CC26"

if not os.path.exists(repo_name):
    print("Cloning repository...")
    subprocess.run(["git", "clone", repo_url], check=True)
    print("Repo berhasil di-clone")
else:
    print("Repo sudah ada")

# Path ke folder Data Clean
BASE_DIR_MAIN = os.path.join(os.getcwd(), repo_name, "Data Clean")

print(f"\nBASE_DIR_MAIN: {BASE_DIR_MAIN}")

# Debug isi folder
if os.path.exists(BASE_DIR_MAIN):
    print("\nIsi folder Data Clean:")
    print(os.listdir(BASE_DIR_MAIN))
else:
    raise FileNotFoundError("Folder 'Data Clean' tidak ditemukan")


Cloning repository...
Repo berhasil di-clone

BASE_DIR_MAIN: /content/CPS-CC26/Data Clean

Isi folder Data Clean:
['jaksel_retail_final_v3.csv', 'jaksel_retail_final_v3 (perlu revisi).csv', 'Luas Wilayah 2024.csv', 'jaksel_pasar_final.csv']


In [ ]:
file_ai = os.path.join(BASE_DIR_MAIN, "jaksel_retail_final_v3.csv")
file_pasar = os.path.join(BASE_DIR_MAIN, "jaksel_pasar_final.csv")

# Validasi file
if not os.path.exists(file_ai):
    raise FileNotFoundError(f"File tidak ditemukan: {file_ai}")

if not os.path.exists(file_pasar):
    raise FileNotFoundError(f"File tidak ditemukan: {file_pasar}")

# Load data
df_ai = pd.read_csv(file_ai)
df_pasar = pd.read_csv(file_pasar)

# Cleaning koordinat
df_ai.dropna(subset=['latitude', 'longitude'], inplace=True)
df_pasar.dropna(subset=['latitude', 'longitude'], inplace=True)

df_ai.reset_index(drop=True, inplace=True)
df_pasar.reset_index(drop=True, inplace=True)

print("\nDATA BERHASIL DIMUAT")
print(f"Total Minimarket        : {len(df_ai)} toko")
print(f"Total Pasar Tradisional: {len(df_pasar)} pasar")


DATA BERHASIL DIMUAT
Total Minimarket        : 662 toko
Total Pasar Tradisional: 23 pasar


In [ ]:
# 3. SPATIAL FEATURES: KOMPETITOR (Alfamart vs Indomaret)
print("[PROSES] Menghitung jarak antar kompetitor dan densitas 500m...")

df_indo = df_ai[df_ai['store'] == 'Indomaret'].reset_index(drop=True)
df_alfa = df_ai[df_ai['store'] == 'Alfamart'].reset_index(drop=True)

coords_indo = list(zip(df_indo['latitude'], df_indo['longitude']))
coords_alfa = list(zip(df_alfa['latitude'], df_alfa['longitude']))
coords_all  = list(zip(df_ai['latitude'], df_ai['longitude']))

jarak_kompetitor_list = []
comp_density_list = []

for i, row in df_ai.iterrows():
    origin = (row['latitude'], row['longitude'])

    # 3A. Hitung Competitor Density 500m
    count_500m = sum(1 for j, target in enumerate(coords_all) if i != j and haversine(origin, target, unit=Unit.METERS) <= 500)
    comp_density_list.append(count_500m)

    # 3B. Hitung Jarak Terdekat Beda Brand
    if row['store'] == 'Alfamart':
        distances = [haversine(origin, t, unit=Unit.METERS) for t in coords_indo]
    else:
        distances = [haversine(origin, t, unit=Unit.METERS) for t in coords_alfa]

    jarak_kompetitor_list.append(round(min(distances), 2) if distances else np.nan)

df_ai['competitor_density_500m'] = comp_density_list
df_ai['jarak_kompetitor_meter'] = jarak_kompetitor_list
df_ai['kompetitor_head_to_head'] = (df_ai['jarak_kompetitor_meter'] <= 50).astype(int)

print("Fitur Jarak Kompetitor dan Densitas berhasil di-generate.")

[PROSES] Menghitung jarak antar kompetitor dan densitas 500m...
Fitur Jarak Kompetitor dan Densitas berhasil di-generate.


In [ ]:
# 4. SPATIAL FEATURES: ZONASI PASAR TRADISIONAL
print("[PROSES] Menghitung jarak ke pasar tradisional...")

jarak_pasar_list = []
pasar_terdekat_list = []

for _, mini in df_ai.iterrows():
    origin = (mini['latitude'], mini['longitude'])
    distances = df_pasar.apply(lambda p: haversine(origin, (p['latitude'], p['longitude']), unit=Unit.METERS), axis=1)

    idx_min = distances.idxmin()
    jarak_pasar_list.append(distances[idx_min])
    pasar_terdekat_list.append(df_pasar.loc[idx_min, 'nama_tempat'])

df_ai['jarak_pasar_meter'] = jarak_pasar_list
df_ai['pasar_terdekat'] = pasar_terdekat_list
df_ai['pelanggaran_zonasi'] = (df_ai['jarak_pasar_meter'] <= 500).astype(int)

print(f"Ditemukan {df_ai['pelanggaran_zonasi'].sum()} toko melanggar zonasi <500m.")

[PROSES] Menghitung jarak ke pasar tradisional...
Ditemukan 99 toko melanggar zonasi <500m.


In [ ]:
# 5. SPATIAL CLUSTERING: K-MEANS & DBSCAN
coords = df_ai[['latitude', 'longitude']].values
coords_rad = np.radians(coords)

# K-Means (10 Kluster untuk zonasi makro/kecamatan)
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df_ai['cluster_kmeans_makro'] = kmeans.fit_predict(coords)

# DBSCAN (Radius 300m, Min 4 sampel untuk deteksi Hotspot/Oversaturasi)
eps_rad = 300 / 6371000 # 300 meter
dbscan = DBSCAN(eps=eps_rad, min_samples=4, algorithm='ball_tree', metric='haversine')
df_ai['cluster_dbscan_hotspot'] = dbscan.fit_predict(coords_rad)

n_hotspots = len(set(df_ai['cluster_dbscan_hotspot'])) - (1 if -1 in df_ai['cluster_dbscan_hotspot'].values else 0)
print(f"Ditemukan {n_hotspots} area Hotspot mikro (radius 300m padat).")

Ditemukan 47 area Hotspot mikro (radius 300m padat).


In [ ]:
# 5. SPATIAL CLUSTERING: K-MEANS & DBSCAN
coords = df_ai[['latitude', 'longitude']].values
coords_rad = np.radians(coords)

# K-Means (10 Kluster untuk zonasi makro/kecamatan)
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
df_ai['cluster_kmeans_makro'] = kmeans.fit_predict(coords)

# DBSCAN (Radius 300m, Min 4 sampel untuk deteksi Hotspot/Oversaturasi)
eps_rad = 300 / 6371000 # 300 meter
dbscan = DBSCAN(eps=eps_rad, min_samples=4, algorithm='ball_tree', metric='haversine')
df_ai['cluster_dbscan_hotspot'] = dbscan.fit_predict(coords_rad)

n_hotspots = len(set(df_ai['cluster_dbscan_hotspot'])) - (1 if -1 in df_ai['cluster_dbscan_hotspot'].values else 0)
print(f"Ditemukan {n_hotspots} area Hotspot mikro (radius 300m padat).")

Ditemukan 47 area Hotspot mikro (radius 300m padat).


In [ ]:
# 6. VALIDASI SCHEMA & EXPORT DATA FINAL
FITUR_AI = [
    'latitude', 'longitude', 'competitor_density_500m',
    'jarak_kompetitor_meter', 'kompetitor_head_to_head',
    'jarak_pasar_meter', 'pelanggaran_zonasi',
    'cluster_kmeans_makro', 'cluster_dbscan_hotspot'
]

# Cek Missing Values
missing_vals = df_ai[FITUR_AI].isna().sum()
if missing_vals.sum() == 0:
    print("Validasi Sukses: Tidak ada missing values pada fitur AI.")

    # Export Dataset
    OUTPUT_PATH = os.path.join(BASE_DIR_MAIN, "jaksel_spatial_features_v4_AI_ready.csv")
    df_ai.to_csv(OUTPUT_PATH, index=False)
    print(f"File V4 tersimpan: {OUTPUT_PATH}")
else:
    print(f"Peringatan! Terdapat missing values:\n{missing_vals[missing_vals > 0]}")

Validasi Sukses: Tidak ada missing values pada fitur AI.
File V4 tersimpan: /content/CPS-CC26/Data Clean/jaksel_spatial_features_v4_AI_ready.csv
